# Xarray-Spatial Resampling: Downscale and upscale rasters

The `resample` function changes a raster's resolution (cell size) without changing its CRS. Use it when you need to match two rasters to a common grid or reduce memory footprint before analysis. Seven interpolation methods cover continuous surfaces, categorical data, and statistical aggregations.

### What you'll build

1. [Generate a synthetic DEM](#Data)
2. [Downsample with `scale_factor`](#Downsample-with-scale_factor)
3. [Upsample with `target_resolution`](#Upsample-with-target_resolution)
4. [Compare resampling methods side by side](#Compare-resampling-methods)
5. [Resample categorical data with `mode`](#Categorical-raster-with-mode)
6. [Run resampling on Dask-backed arrays](#Works-with-Dask)

![Resample preview](images/resample_preview.png)

[Downsample](#Downsample-with-scale_factor) · [Upsample](#Upsample-with-target_resolution) · [Method comparison](#Compare-resampling-methods) · [Categorical mode](#Categorical-raster-with-mode) · [Dask](#Works-with-Dask)

Standard imports plus the `resample` function and terrain generator.

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from xrspatial import resample
from xrspatial.terrain import generate_terrain

## Data

Generate a 200x200 synthetic terrain raster with 0.5 m resolution. The coordinate grid runs from 0 to 100 in both axes.

In [ ]:
dem = generate_terrain(width=200, height=200)
# Assign a regular coordinate grid
dem = dem.assign_coords(
    y=np.linspace(100, 0, dem.sizes['y']),
    x=np.linspace(0, 100, dem.sizes['x']),
)
dem.attrs['res'] = (0.5, 0.5)

print(f'Shape: {dem.shape}, resolution: {dem.attrs["res"]}')

The DEM covers a 100x100 m area at 0.5 m cell size. Below is the elevation surface plotted with the `terrain` colormap.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
dem.plot.imshow(ax=ax, cmap='terrain')
ax.set_title(f'Original DEM ({dem.shape[0]}x{dem.shape[1]}, res={dem.attrs["res"][0]:.1f}m)')
plt.tight_layout()

## Downsample with `scale_factor`

A `scale_factor` below 1.0 reduces the raster dimensions. Here we shrink by 4x using bilinear interpolation.

| Method | Direction | Best for |
|--------|-----------|----------|
| `nearest` | up/down | Categorical data, fast preview |
| `bilinear` | up/down | Smooth continuous surfaces |
| `cubic` | up/down | High-quality continuous surfaces |
| `average` | down only | Aggregating high-res to low-res |
| `min`, `max` | down only | Extremes within each output cell |
| `median` | down only | Robust centre, ignores outliers |
| `mode` | down only | Majority class in categorical rasters |

In [ ]:
down = resample(dem, scale_factor=0.25, method='bilinear')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
dem.plot.imshow(ax=axes[0], cmap='terrain')
axes[0].set_title(f'Original ({dem.shape[0]}x{dem.shape[1]})')
down.plot.imshow(ax=axes[1], cmap='terrain')
axes[1].set_title(f'Downsampled 4x ({down.shape[0]}x{down.shape[1]})')
plt.tight_layout()

## Upsample with `target_resolution`

Instead of a scale factor, you can specify the desired output cell size directly. Here we upsample the coarse raster back to 0.5 m using cubic interpolation.

In [ ]:
up = resample(down, target_resolution=0.5, method='cubic')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
down.plot.imshow(ax=axes[0], cmap='terrain')
axes[0].set_title(f'Coarse ({down.shape[0]}x{down.shape[1]})')
up.plot.imshow(ax=axes[1], cmap='terrain')
axes[1].set_title(f'Upsampled to 0.5m ({up.shape[0]}x{up.shape[1]})')
plt.tight_layout()

## Compare resampling methods

Four methods applied to the same 10x downsample. Nearest keeps sharp edges but shows staircase artifacts. Bilinear and cubic produce smoother results. Average computes the mean of all input cells that fall within each output cell.

In [ ]:
methods = ['nearest', 'bilinear', 'cubic', 'average']
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

for ax, method in zip(axes, methods):
    out = resample(dem, scale_factor=0.1, method=method)
    out.plot.imshow(ax=ax, cmap='terrain', add_colorbar=False)
    ax.set_title(method)
    ax.set_aspect('equal')

plt.suptitle('Downsample 10x with different methods', y=1.02)
plt.tight_layout()

## Categorical raster with `mode`

For classified rasters, `mode` picks the most frequent value within each output cell. This preserves class boundaries better than any interpolation method, which would produce fractional class values.

In [ ]:
from xrspatial import equal_interval

# Classify elevation into 5 zones
classes = equal_interval(dem, k=5)
classes.attrs = dem.attrs.copy()
classes = classes.assign_coords(dem.coords)

# Downsample: mode preserves class boundaries
classes_down = resample(classes.astype('float32'),
                        scale_factor=0.2, method='mode')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
classes.plot.imshow(ax=axes[0], cmap='Set2')
axes[0].set_title(f'Classes ({classes.shape[0]}x{classes.shape[1]})')
classes_down.plot.imshow(ax=axes[1], cmap='Set2')
axes[1].set_title(f'Mode downsample ({classes_down.shape[0]}x{classes_down.shape[1]})')
plt.tight_layout()

## Works with Dask

Pass a Dask-backed DataArray and `resample` returns a lazy result. Each chunk is resampled independently.

In [ ]:
import dask.array as da

dask_dem = dem.copy()
dask_dem.data = da.from_array(dem.values, chunks=(100, 100))

result = resample(dask_dem, scale_factor=0.5, method='bilinear')
print(f'Input:  {dask_dem.shape} (dask, chunks={dask_dem.data.chunksize})')
print(f'Output: {result.shape} (dask, chunks={result.data.chunksize})')
print(f'Computed shape: {result.compute().shape}')

### References

- [Raster resampling overview](https://desktop.arcgis.com/en/arcmap/latest/extensions/spatial-analyst/performing-analysis/cell-size-and-resampling-in-analysis.htm), Esri ArcGIS documentation
- [Resampling methods comparison](https://gisgeography.com/raster-resampling/), GIS Geography
- [xarray.DataArray](https://docs.xarray.dev/en/stable/generated/xarray.DataArray.html), xarray documentation